# Parte 3 — Ecuaciones elípticas
## 3.2 Propiedades de las funciones armónicas
### 3.2.02 Regularidad, desigualdad de Harnack, estimaciones interiores, Liouville y analiticidad

**Fuente principal:** parte izquierda y superior de la página 8 de las notas manuscritas
del archivo `Ecuación De Onda(1).pdf`.

Este notebook continúa la numeración de
`03.2.01_Propiedad_del_promedio_y_subarmonicidad.ipynb`.

Se trabajan únicamente los resultados anteriores al encabezado
“Función de Green y fórmula de Poisson”:

1. regularidad a partir de la propiedad del valor promedio;
2. desigualdad de Harnack;
3. estimaciones interiores del gradiente y de derivadas;
4. teorema de Liouville;
5. analiticidad de las funciones armónicas.

El bloque de función de Green de la parte derecha de la página queda para el
siguiente notebook.

## Convenciones editoriales

- **Transcripción literal de las notas:** conserva el orden y la notación legible.
- **Lectura incompleta:** marca una expresión que la página no desarrolla por completo.
- **Aclaración:** explicita una hipótesis necesaria.
- **Complemento:** añade una demostración clásica para cerrar un hueco.
- Toda la matemática usa exclusivamente `$...$` y `$$...$$`.

## Estado de las fuentes

La imagen manuscrita antes enlazada como `pagina_08_notas.png` no está incluida en el repositorio. El desarrollo que sigue es autocontenido. Para cotejar el enunciado oficial debe consultarse el PDF original fuera de este repositorio; no se sustituye aquí por una imagen inventada.


# Simulaciones y visualizaciones

Las celdas se ejecutan directamente: no existe una bandera `VIDEO=True`.

Se incluyen:

1. animación del suavizamiento interior de datos de frontera rugosos;
2. barrido Monte Carlo/GPU de la desigualdad local de Harnack;
3. visualización de estimaciones del gradiente en bolas concéntricas;
4. decaimiento de la cota utilizada en Liouville;
5. convergencia exponencial de aproximaciones de Taylor.

CuPy/CUDA se usa automáticamente cuando hay un dispositivo disponible y aporta
valor en los barridos y mallas densas.

In [ ]:
from __future__ import annotations

import math
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import Video, Image, display

BASE = Path("..")
FIG_DIR = BASE / "figuras"
ANIM_DIR = BASE / "animaciones"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANIM_DIR.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = False

try:
    import cupy as cp

    if cp.cuda.runtime.getDeviceCount() > 0:
        xp = cp
        GPU_AVAILABLE = True
        print("Backend numérico: CuPy/CUDA")
    else:
        raise RuntimeError("No se encontró un dispositivo CUDA.")
except Exception as exc:
    xp = np
    print("Backend numérico: NumPy/CPU")
    print("CuPy no disponible:", type(exc).__name__)


def to_cpu(array):
    if GPU_AVAILABLE:
        return cp.asnumpy(array)
    return np.asarray(array)


def save_and_display_animation(
    animation,
    stem,
    fps=60,
    dpi=150,
    bitrate=10000,
):
    """Guarda y muestra automáticamente una animación."""
    if shutil.which("ffmpeg"):
        output = ANIM_DIR / f"{stem}.mp4"
        writer = FFMpegWriter(
            fps=fps,
            bitrate=bitrate,
            metadata={"title": stem},
        )
        animation.save(output, writer=writer, dpi=dpi)
        display(Video(str(output), embed=True))
    else:
        output = ANIM_DIR / f"{stem}.gif"
        writer = PillowWriter(fps=min(fps, 35))
        animation.save(
            output,
            writer=writer,
            dpi=min(dpi, 110),
        )
        display(Image(filename=str(output)))

    print("Animación guardada en:", output.resolve())
    return output

## Simulación 3.2.D — Suavizamiento interior de datos de frontera

En el disco unitario, un dato de frontera con serie de Fourier

$$
g(\theta)
=
a_0+
\sum_{m=1}^{M}
\left(
a_m\cos(m\theta)+b_m\sin(m\theta)
\right)
$$

tiene extensión armónica

$$
u(r,\theta)
=
a_0+
\sum_{m=1}^{M}
r^m
\left(
a_m\cos(m\theta)+b_m\sin(m\theta)
\right).
$$

Cuando $r<1$, los modos de alta frecuencia se multiplican por $r^m$ y se
amortiguan rápidamente. La animación recorre círculos concéntricos desde la
frontera hacia el centro.

In [ ]:
rng = np.random.default_rng(20260721)

M = 90
n_theta = 2400 if GPU_AVAILABLE else 1200
theta = xp.linspace(
    0.0,
    2.0 * math.pi,
    n_theta,
    endpoint=False,
)
modes = xp.arange(1, M + 1, dtype=xp.float64)

a_cpu = rng.normal(size=M) / np.sqrt(np.arange(1, M + 1))
b_cpu = rng.normal(size=M) / np.sqrt(np.arange(1, M + 1))
a = xp.asarray(a_cpu)
b = xp.asarray(b_cpu)

cos_matrix = xp.cos(modes[:, None] * theta[None, :])
sin_matrix = xp.sin(modes[:, None] * theta[None, :])

boundary = xp.sum(
    a[:, None] * cos_matrix
    + b[:, None] * sin_matrix,
    axis=0,
)
scale = xp.max(xp.abs(boundary))

radii = np.linspace(1.0, 0.04, 220)
profiles = []

for radius in radii:
    damping = xp.asarray(radius) ** modes
    profile = xp.sum(
        damping[:, None]
        * (
            a[:, None] * cos_matrix
            + b[:, None] * sin_matrix
        ),
        axis=0,
    )
    profiles.append(
        to_cpu(profile / scale).astype(np.float32)
    )

profiles = np.asarray(profiles)
theta_cpu = to_cpu(theta)

print("Número de modos:", M)
print("Número de puntos angulares:", n_theta)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

line, = ax.plot(theta_cpu, profiles[0])
ax.set_xlim(0.0, 2.0 * math.pi)
ax.set_ylim(
    float(np.min(profiles)) - 0.05,
    float(np.max(profiles)) + 0.05,
)
ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$u(r,\theta)$")
ax.set_title("Suavizamiento de la extensión armónica")

status = ax.text(
    0.02,
    0.96,
    "",
    transform=ax.transAxes,
    va="top",
)


def update_smoothing(frame):
    line.set_data(theta_cpu, profiles[frame])
    status.set_text(rf"$r={radii[frame]:.3f}$")
    return line, status


animation = FuncAnimation(
    fig,
    update_smoothing,
    frames=len(radii),
    interval=1000.0 / 60.0,
    blit=False,
)

fig.tight_layout()

save_and_display_animation(
    animation,
    "03.2.D_suavizamiento_armonico",
    fps=60,
    dpi=160,
    bitrate=12000,
)

plt.close(fig)

## Simulación 3.2.E — Barrido numérico de la desigualdad local de Harnack

En dimensión $n=2$, la versión de las notas afirma que si

$$
|x-y|=r_0>0,
\qquad
\overline{B_{2r_0}(x)}\subset\Omega,
$$

y $u\geq0$ satisface la propiedad del promedio, entonces

$$
u(y)\leq2^2u(x)=4u(x).
$$

Se toma la función armónica positiva

$$
u(x_1,x_2)=2+x_1
$$

en el disco unitario y se muestrean muchos pares admisibles.

In [ ]:
n_samples = 1_500_000 if GPU_AVAILABLE else 180_000
rng_xp = (
    xp.random.default_rng(20260721)
    if GPU_AVAILABLE
    else np.random.default_rng(20260721)
)

radius_x = 0.55 * xp.sqrt(rng_xp.random(n_samples))
angle_x = 2.0 * math.pi * rng_xp.random(n_samples)

x1 = radius_x * xp.cos(angle_x)
x2 = radius_x * xp.sin(angle_x)

distance_to_boundary = 1.0 - xp.sqrt(x1**2 + x2**2)
r0 = 0.48 * distance_to_boundary * rng_xp.random(n_samples)

angle_y = 2.0 * math.pi * rng_xp.random(n_samples)
y1 = x1 + r0 * xp.cos(angle_y)
y2 = x2 + r0 * xp.sin(angle_y)

u_x = 2.0 + x1
u_y = 2.0 + y1
ratios = u_y / u_x

r0_cpu = to_cpu(r0)
ratios_cpu = to_cpu(ratios)

print("Número de pares:", n_samples)
print("Máximo cociente u(y)/u(x):", float(np.max(ratios_cpu)))
print("Cota de las notas en n=2:", 4.0)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

selection = np.linspace(
    0,
    len(ratios_cpu) - 1,
    min(25000, len(ratios_cpu)),
    dtype=int,
)

ax.scatter(
    r0_cpu[selection],
    ratios_cpu[selection],
    s=4,
    alpha=0.25,
)
ax.axhline(4.0, linestyle="--", label=r"$2^n=4$")
ax.set_xlabel(r"$r_0=|x-y|$")
ax.set_ylabel(r"$u(y)/u(x)$")
ax.set_title("Barrido numérico de la desigualdad local de Harnack")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.2.E_barrido_harnack.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

## Simulación 3.2.F — Estimaciones del gradiente en bolas concéntricas

Se construye un polinomio armónico en el disco,

$$
u(r,\theta)
=
a_0+
\sum_{m=1}^{M}
r^m
\left(
a_m\cos(m\theta)+b_m\sin(m\theta)
\right).
$$

Para $0<\rho<R=1$ se calcula

$$
Q(\rho)
=
\frac{(R-\rho)
\displaystyle\max_{B_\rho}|\nabla u|}
{\displaystyle\max_{B_R}|u|}.
$$

Las estimaciones interiores predicen que $Q(\rho)$ permanece acotado por una
constante que depende sólo de la dimensión.

In [ ]:
M_grad = 18
rng_grad = np.random.default_rng(314159)

a_grad = rng_grad.normal(size=M_grad) / (
    np.arange(1, M_grad + 1) ** 1.2
)
b_grad = rng_grad.normal(size=M_grad) / (
    np.arange(1, M_grad + 1) ** 1.2
)

n_r = 620 if GPU_AVAILABLE else 320
n_t = 1200 if GPU_AVAILABLE else 600

r_grid = xp.linspace(0.0, 1.0, n_r)
t_grid = xp.linspace(
    0.0,
    2.0 * math.pi,
    n_t,
    endpoint=False,
)

RR, TT = xp.meshgrid(r_grid, t_grid, indexing="ij")
m = xp.arange(1, M_grad + 1, dtype=xp.float64)

a_g = xp.asarray(a_grad)
b_g = xp.asarray(b_grad)

powers = RR[:, :, None] ** m[None, None, :]
cos_terms = xp.cos(TT[:, :, None] * m[None, None, :])
sin_terms = xp.sin(TT[:, :, None] * m[None, None, :])

U = xp.sum(
    powers
    * (
        a_g[None, None, :] * cos_terms
        + b_g[None, None, :] * sin_terms
    ),
    axis=2,
)

safe_R = xp.maximum(RR, 1e-12)

Ur = xp.sum(
    m[None, None, :]
    * safe_R[:, :, None] ** (m[None, None, :] - 1.0)
    * (
        a_g[None, None, :] * cos_terms
        + b_g[None, None, :] * sin_terms
    ),
    axis=2,
)

Ut = xp.sum(
    powers
    * m[None, None, :]
    * (
        -a_g[None, None, :] * sin_terms
        + b_g[None, None, :] * cos_terms
    ),
    axis=2,
)

grad_norm = xp.sqrt(Ur**2 + (Ut / safe_R) ** 2)
grad_norm[0, :] = xp.sqrt(a_g[0] ** 2 + b_g[0] ** 2)

sup_u = float(to_cpu(xp.max(xp.abs(U))))

rho_values = np.linspace(0.05, 0.96, 70)
q_values = []
r_cpu = to_cpu(r_grid)

for rho in rho_values:
    index = int(np.searchsorted(r_cpu, rho, side="right"))
    grad_sup = float(
        to_cpu(xp.max(grad_norm[:index, :]))
    )
    q_values.append(
        (1.0 - rho) * grad_sup / sup_u
    )

q_values = np.asarray(q_values)

print("sup_{B_1}|u| =", sup_u)
print("máximo observado de Q(rho) =", np.max(q_values))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(rho_values, q_values, marker="o", ms=3)
ax.set_xlabel(r"$\rho$")
ax.set_ylabel(r"$Q(\rho)$")
ax.set_title("Escalamiento de la estimación interior del gradiente")
ax.grid(True, alpha=0.3)
fig.tight_layout()

path = FIG_DIR / "03.2.F_estimacion_gradiente.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

## Simulación 3.2.G — Liouville y crecimiento de las bolas

Si $u$ es armónica y globalmente acotada por $M$, la estimación interior implica

$$
|\nabla u(x_0)|
\leq
\frac{C_nM}{R}
$$

para todo $R>0$. Al hacer $R\to\infty$, el lado derecho tiende a cero.

In [ ]:
R_values = np.geomspace(0.5, 1000.0, 500)

fig, ax = plt.subplots(figsize=(9, 6))

for constant in (1.0, 2.0, 5.0, 10.0):
    ax.loglog(
        R_values,
        constant / R_values,
        label=rf"$C_nM={constant:g}$",
    )

ax.set_xlabel(r"$R$")
ax.set_ylabel(r"$C_nM/R$")
ax.set_title("La cota del gradiente desaparece cuando $R\to\infty$")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.2.G_liouville_cota.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

## Simulación 3.2.H — Convergencia de series de Taylor

La función

$$
u(r,\theta)
=
1+2\sum_{m=1}^{\infty}r^m\cos(m\theta)
$$

es armónica para $0\leq r<1$. Para un radio fijo $\rho<1$, el error de truncar
la serie después de $N$ modos decrece geométricamente.

In [ ]:
theta_taylor = np.linspace(
    0.0,
    2.0 * math.pi,
    3000,
    endpoint=False,
)

rho_values_taylor = (0.35, 0.60, 0.82)
orders = np.arange(0, 81)
errors_by_radius = {}

for rho in rho_values_taylor:
    exact = (
        1.0 - rho**2
    ) / (
        1.0
        - 2.0 * rho * np.cos(theta_taylor)
        + rho**2
    )

    errors = []

    for order in orders:
        approximation = np.ones_like(theta_taylor)

        for mode in range(1, order + 1):
            approximation += (
                2.0
                * rho**mode
                * np.cos(mode * theta_taylor)
            )

        errors.append(
            np.max(np.abs(exact - approximation))
        )

    errors_by_radius[rho] = np.asarray(errors)

fig, ax = plt.subplots(figsize=(9, 6))

for rho, errors in errors_by_radius.items():
    ax.semilogy(
        orders,
        errors,
        marker="o",
        ms=3,
        label=rf"$\rho={rho}$",
    )

ax.set_xlabel("orden de truncamiento")
ax.set_ylabel("error máximo")
ax.set_title("Convergencia de la expansión armónica")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.2.H_analiticidad_taylor.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

# 3.2.4 Regularidad

### **Transcripción literal de las notas**

**Teorema I.**

Sea $\Omega\subset\mathbb R^n$ abierto y sea $u\in C(\Omega)$ una función
continua que satisface la P.V.P. Entonces

$$
u\in C^\infty(\Omega).
$$

Aquí P.V.P. denota la propiedad del valor promedio.

## **Teorema 3.2.8 (Regularidad a partir de la propiedad del valor promedio).**

Sea $\Omega\subset\mathbb R^n$ abierto y sea $u\in C(\Omega)$. Supongamos que
para toda bola cerrada $\overline{B_r(x)}\subset\Omega$,

$$
u(x)
=
\frac{1}{|\partial B_r|}
\int_{\partial B_r(x)}u(y)\,dS_y.
$$

Entonces

$$
u\in C^\infty(\Omega).
$$

En particular, $u$ es armónica en $\Omega$.

### Demostración añadida

Sea $\eta\in C_c^\infty(B_1(0))$ una función radial, no negativa, tal que

$$
\int_{\mathbb R^n}\eta(z)\,dz=1.
$$

Para $\varepsilon>0$ definimos

$$
\eta_\varepsilon(z)
=
\varepsilon^{-n}\eta(z/\varepsilon).
$$

Fijemos $\Omega'\Subset\Omega$ y elijamos $\varepsilon$ menor que la distancia
de $\Omega'$ a $\partial\Omega$.

Como $\eta$ es radial, la convolución es una superposición ponderada de
promedios esféricos centrados en $x$. Por la P.V.P., para $x\in\Omega'$,

$$
(u*\eta_\varepsilon)(x)=u(x).
$$

El miembro izquierdo es de clase $C^\infty$. Como $\Omega'\Subset\Omega$ es
arbitrario, $u\in C^\infty(\Omega)$.

El recíproco de la propiedad del promedio, demostrado en el notebook anterior,
implica que $\Delta u=0$.

$\square$

### Ejercicios — Sección 3.2.4

> **Ruta corta de estudio:** dos ejercicios representativos; este subtema se integra después en los problemas prioritarios.

1. Escriba en coordenadas polares todos los pasos que justifican
   $u*\eta_\varepsilon=u$.

2. Demuestre la misma regularidad partiendo de la propiedad del promedio sobre bolas.


# 3.2.5 Desigualdad de Harnack

### **Transcripción literal de las notas**

**Lema II (Desigualdad de Harnack).**

Sea $\Omega\subset\mathbb R^n$ abierto. Sea $u\in C(\Omega)$, $u\geq0$, con
la P.V.P.

Sean $x,y\in\Omega$ tales que

$$
|x-y|=r_0>0,
\qquad
x\neq y,
$$

y

$$
\overline{B_{2r_0}(x)}\subset\Omega.
$$

Entonces

$$
u(y)\leq2^nu(x).
$$

## **Lema 3.2.9 (Desigualdad local de Harnack).**

Bajo las hipótesis anteriores,

$$
u(y)\leq2^nu(x).
$$

### Demostración añadida

Como $|x-y|=r_0$,

$$
B_{r_0}(y)\subset B_{2r_0}(x).
$$

La no negatividad de $u$ y la propiedad del promedio sobre bolas dan

$$
\begin{aligned}
u(y)
&=
\frac{1}{|B_{r_0}|}
\int_{B_{r_0}(y)}u(z)\,dz\\
&\leq
\frac{1}{|B_{r_0}|}
\int_{B_{2r_0}(x)}u(z)\,dz\\
&=
2^nu(x).
\end{aligned}
$$

$\square$

## **Teorema 3.2.10 (Desigualdad de Harnack en compactos).**

Sea $\Omega$ un dominio y sea $K\Subset\Omega$ compacto y conexo. Existe una
constante $C=C(n,K,\Omega)$ tal que toda función armónica no negativa satisface

$$
\sup_Ku\leq C\inf_Ku.
$$

### Complemento

La prueba encadena un número finito y uniforme de bolas contenidas en $\Omega$ y
aplica repetidamente el Lema 3.2.9.

## **Corolario 3.2.11 (Dicotomía de positividad).**

Si $u\geq0$ es armónica en un dominio, entonces $u\equiv0$ o $u>0$.

### Ejercicios — Sección 3.2.5

> **Ruta corta de estudio:** dos ejercicios representativos; este subtema se integra después en los problemas prioritarios.

1. Reproduzca la prueba usando promedios sobre esferas.

2. Obtenga una cota de Harnack en $B_{1/2}$ para funciones positivas armónicas
   en $B_1$.


# 3.2.6 Estimaciones interiores del gradiente

### **Transcripción literal de las notas**

**Teorema III (Estimaciones del gradiente).**

Sea

$$
u\in C(\overline{B_R(x_0)})
$$

armónica, con $x_0\in\Omega$, $\Omega\subset\mathbb R^n$ abierto y $R>0$.

> **Lectura incompleta.** La página conserva el encabezado y las hipótesis, pero
> no una fórmula visible para la estimación.

## **Teorema 3.2.12 (Estimación interior del gradiente).**

Existe $C_n>0$, dependiente sólo de la dimensión, tal que

$$
|\nabla u(x_0)|
\leq
\frac{C_n}{R}
\max_{\overline{B_R(x_0)}}|u|.
$$

Más generalmente, para $0<\rho<R$,

$$
\max_{\overline{B_\rho(x_0)}}|\nabla u|
\leq
\frac{C_n}{R-\rho}
\max_{\overline{B_R(x_0)}}|u|.
$$

### Demostración añadida

Se usa una regularización radial $\eta_R$. La propiedad del promedio implica
$u=u*\eta_R$ cerca de $x_0$. Entonces

$$
\nabla u(x_0)
=
\int u(y)\nabla\eta_R(x_0-y)\,dy.
$$

Como

$$
\int|\nabla\eta_R|
=
\frac{1}{R}\int|\nabla\eta|,
$$

se obtiene la primera estimación. La segunda se aplica punto por punto usando
bolas de radio $R-\rho$.

$\square$

## **Corolario 3.2.13 (Estimaciones de derivadas).**

Para cada multiíndice $\alpha$,

$$
|D^\alpha u(x_0)|
\leq
\frac{C_{n,\alpha}}{R^{|\alpha|}}
\max_{\overline{B_R(x_0)}}|u|.
$$

Además, pueden elegirse constantes con crecimiento factorial:

$$
C_{n,\alpha}
\leq
A_n^{|\alpha|}|\alpha|!.
$$

### Ejercicios — Sección 3.2.6

> **Ruta corta de estudio:** dos ejercicios representativos; este subtema se integra después en los problemas prioritarios.

1. Compruebe la estimación para $u(x,y)=x^2-y^2$.

2. Demuestre la versión en bolas concéntricas para derivadas de orden arbitrario.


# 3.2.7 Teorema de Liouville

### **Transcripción literal de las notas**

**Teorema de Liouville.**

Sea

$$
u:\mathbb R^n\longrightarrow\mathbb R
$$

armónica en $\mathbb R^n$ y uniformemente acotada. Es decir, existe $C>0$ tal que

$$
|u(x)|\leq C
$$

para todo $x\in\mathbb R^n$. Entonces $u$ es constante.

## **Teorema 3.2.14 (Teorema de Liouville para funciones armónicas).**

Toda función armónica y acotada en $\mathbb R^n$ es constante.

### Demostración añadida

Si $|u|\leq M$, para todo $R>0$,

$$
|\nabla u(x_0)|
\leq
\frac{C_nM}{R}.
$$

Haciendo $R\to\infty$ se obtiene $\nabla u(x_0)=0$. Como $x_0$ es arbitrario,
$u$ es constante.

$\square$

### Ejercicios — Sección 3.2.7

> **Ruta corta de estudio:** dos ejercicios representativos; este subtema se integra después en los problemas prioritarios.

1. Demuestre Liouville usando Harnack.

2. Pruebe que una función armónica entera con crecimiento $o(|x|)$ es constante.


# 3.2.8 Analiticidad

### **Transcripción literal de las notas**

**Teorema.**

Sea $u\in C^1(\Omega)$ armónica en $\Omega\subset\mathbb R^n$ abierto. Entonces
$u$ es analítica en $\Omega$.

### **Aclaración.**

En la definición clásica, “armónica” presupone $C^2$. El teorema de regularidad
muestra que incluso una función continua con la P.V.P. es $C^\infty$.

## **Teorema 3.2.15 (Analiticidad de las funciones armónicas).**

Para cada $x_0\in\Omega$ existe $r>0$ tal que

$$
u(x)
=
\sum_{\alpha\in\mathbb N_0^n}
\frac{D^\alpha u(x_0)}{\alpha!}
(x-x_0)^\alpha
$$

para $x\in B_r(x_0)$.

### Demostración añadida

Las estimaciones factoriales proporcionan

$$
|D^\alpha u(x_0)|
\leq
\frac{A_n^{|\alpha|}|\alpha|!}{R^{|\alpha|}}
\max_{B_R(x_0)}|u|.
$$

La serie de Taylor converge para $A_n|x-x_0|<R$. La fórmula de Taylor con
resto y la misma cota muestran que el resto tiende a cero.

$\square$

### Ejercicios — Sección 3.2.8

> **Ruta corta de estudio:** dos ejercicios representativos; este subtema se integra después en los problemas prioritarios.

1. Complete la estimación del resto de Taylor.

2. Demuestre que si todas las derivadas de una función armónica se anulan en un
   punto, entonces la función se anula en una vecindad.


# Control de cobertura y estado del capítulo

## Contenido cubierto

- regularidad $C^\infty$ desde la propiedad del promedio;
- Harnack local y en compactos;
- estimaciones interiores del gradiente y de derivadas;
- Liouville;
- analiticidad y continuación única local.

## Complementos añadidos

La página enuncia los resultados, pero no desarrolla las demostraciones y la
fórmula de la estimación del gradiente no es visible. Se incorporaron las
versiones clásicas, marcadas como complementos.

## Pendiente inmediato

No se comenzó el bloque de la parte derecha de la página 8:

- función de Green;
- parte regular;
- fórmula de representación;
- identidad de Green asociada.

El siguiente notebook será

$$
\texttt{03.4.01\_Funcion\_de\_Green\_y\_formula\_de\_representacion.ipynb}.
$$